# SceneHawk — Feelings Classifier (Colab)

Fine-tune a **BERT-style** multi-label classifier to detect viewer feelings in film
reviews, report **F1** (your numbers vs a general LLM), and save the model.

**Before you run:** `Runtime -> Change runtime type -> GPU` (a free T4 is plenty).

It runs end-to-end on public **GoEmotions** data out of the box, so you can see the
pipeline work today. When your labeled reviews exist, switch to **Option B** in the
data cell — nothing else changes.

> Colab runtimes are temporary. Use the **Save** cell to push the model to Drive or the
> Hugging Face Hub, or you lose it when the session ends.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip install -q -U transformers datasets evaluate accelerate scikit-learn

## 3. Config

We start from **`SamLowe/roberta-base-go_emotions`** — its encoder is already tuned on
emotion data, so it is a better starting point than plain `roberta-base` for this task.

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.metrics import f1_score, classification_report

# ---- Config ----
BASE_MODEL = "SamLowe/roberta-base-go_emotions"   # emotion-tuned encoder = warm start
RESET_HEAD = True             # our feelings are NOT the checkpoint's labels -> fresh head
MAX_LEN    = 256              # reviews are longer than tweets
EPOCHS     = 3
LR         = 2e-5
BATCH      = 16
THRESHOLD  = 0.5              # a feeling is "present" if sigmoid prob >= THRESHOLD
OUTPUT_DIR = "scenehawk-feelings"

print("CUDA available:", torch.cuda.is_available())

## 4. Load data

**Option A (default):** GoEmotions — runs now, proves the pipeline.
**Option B:** your labeled film reviews — uncomment once Week 3 is done.

In [ ]:
# Your Week 1 feelings list (used by Option B). Update if Thursday's test changes it.
FEELINGS_28 = [
    "tense", "dread", "scared", "unsettled", "disturbed",
    "sad", "grief-stricken", "lonely", "moved", "bittersweet",
    "cozy", "heartwarming", "tender", "hopeful", "uplifted", "joyful",
    "thrilled", "amused", "awed", "curious",
    "thoughtful", "nostalgic", "inspired", "confused",
    "uncomfortable", "angry", "disgusted", "drained",
]

# ---- OPTION A: GoEmotions demo (runs out of the box) ----
raw = load_dataset("go_emotions", "simplified")
LABELS = raw["train"].features["labels"].feature.names
id2name = dict(enumerate(LABELS))
def _idx_to_names(ex): return {"feelings": [id2name[i] for i in ex["labels"]]}
train_raw = raw["train"].map(_idx_to_names)
val_raw   = raw["validation"].map(_idx_to_names)
test_raw  = raw["test"].map(_idx_to_names)
TEXT_COL  = "text"
print(len(LABELS), "labels:", LABELS)

# ---- OPTION B: your labeled film reviews ----
# Each row: {"text": "a slow, dread-soaked ghost story ...", "feelings": ["dread", "lonely"]}
#
# LABELS = FEELINGS_28
# data = load_dataset("json", data_files={"train": "train.jsonl",
#                                         "validation": "val.jsonl",
#                                         "test": "test.jsonl"})
# train_raw, val_raw, test_raw = data["train"], data["validation"], data["test"]
# TEXT_COL = "text"

## 5. Tokenize + build multi-hot labels

In [ ]:
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
num_labels = len(LABELS)

tok = AutoTokenizer.from_pretrained(BASE_MODEL)

def encode(ex):
    enc = tok(ex[TEXT_COL], truncation=True, max_length=MAX_LEN)
    vec = np.zeros(num_labels, dtype=np.float32)
    for f in ex["feelings"]:
        if f in label2id:
            vec[label2id[f]] = 1.0
    enc["labels"] = vec
    return enc

train_ds = train_raw.map(encode, remove_columns=train_raw.column_names)
val_ds   = val_raw.map(encode,   remove_columns=val_raw.column_names)
test_ds  = test_raw.map(encode,  remove_columns=test_raw.column_names)
print(train_ds)

## 6. Model (multi-label head)

**Why the head reset matters:** your 28 feelings and GoEmotions' 28 labels are *different
sets that happen to be the same size*. Without resetting, the old head loads silently with
the wrong meanings — it would still train, but from confidently wrong associations.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=num_labels,
    problem_type="multi_label_classification",   # -> sigmoid + BCE loss
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

if RESET_HEAD:
    model.classifier.dense.reset_parameters()
    model.classifier.out_proj.reset_parameters()
    print(f"classification head reinitialized for our {num_labels} feelings")

## 7. Metrics (F1 — your headline numbers)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))            # sigmoid
    preds = (probs >= THRESHOLD).astype(int)
    return {
        "f1_micro":    f1_score(labels, preds, average="micro",    zero_division=0),
        "f1_macro":    f1_score(labels, preds, average="macro",    zero_division=0),
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),
    }

## 8. Train

In [ ]:
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LR,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",          # older transformers: evaluation_strategy="epoch"
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tok,
    data_collator=DataCollatorWithPadding(tok),
    compute_metrics=compute_metrics,
)
trainer.train()

## 9. Evaluate + per-feeling report

The per-feeling table is what tells you which feelings to merge, split or cut in Week 4.

In [ ]:
print("Validation:", trainer.evaluate(val_ds))
print("Test:      ", trainer.evaluate(test_ds))

pred  = trainer.predict(test_ds)
probs = 1 / (1 + np.exp(-pred.predictions))
preds = (probs >= THRESHOLD).astype(int)
print(classification_report(pred.label_ids, preds, target_names=LABELS, zero_division=0))

## 10. Save the model (before the session ends!)

In [ ]:
trainer.save_model(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)

# --- Persist to Google Drive ---
# from google.colab import drive; drive.mount("/content/drive")
# import shutil
# shutil.copytree(OUTPUT_DIR, f"/content/drive/MyDrive/{OUTPUT_DIR}", dirs_exist_ok=True)

# --- Or push to the Hugging Face Hub ---
# from huggingface_hub import notebook_login; notebook_login()
# model.push_to_hub("your-username/scenehawk-feelings")
# tok.push_to_hub("your-username/scenehawk-feelings")

## 11. Try it on a sample review

In [ ]:
from transformers import pipeline
clf = pipeline("text-classification", model=OUTPUT_DIR, tokenizer=OUTPUT_DIR,
               top_k=None, function_to_apply="sigmoid")

sample = ("A slow, quiet film about loneliness that looks cold and grey, "
          "full of dread but with no violence.")
for r in sorted(clf(sample)[0], key=lambda x: -x["score"])[:6]:
    print(f"{r['label']:>14}: {r['score']:.2f}")

## 12. Compare to a general LLM (RQ1)

Run a general LLM **zero-shot** on the SAME test reviews, turn its answers into multi-hot
vectors, and score them with the SAME `f1_score` call. Report:

| System | F1 (micro) |
|---|---|
| Our fine-tuned model | ... |
| `SamLowe/roberta-base-go_emotions` off the shelf | ... |
| Zero-shot general LLM | ... |

In [ ]:
# def llm_predict(text: str) -> list[str]:
#     """Ask a general LLM to pick feelings from LABELS; return a JSON list of names."""
#     ...
#
# llm_preds = np.zeros((len(test_raw), num_labels), dtype=int)
# for i, t in enumerate(test_raw[TEXT_COL]):
#     for f in llm_predict(t):
#         if f in label2id:
#             llm_preds[i, label2id[f]] = 1
#
# print("LLM  f1_micro:", f1_score(pred.label_ids, llm_preds, average="micro", zero_division=0))
# print("Ours f1_micro:", f1_score(pred.label_ids, preds,     average="micro", zero_division=0))